In [1]:
import xarray as xr
import numpy as np

DATA_URL = "https://pae-paha.pacioos.hawaii.edu/thredds/dodsC/dhw_5km"

print("Mencoba menghubungkan ke server...")
try:
    ds = xr.open_dataset(DATA_URL, engine='netcdf4')
    print("Berhasil terhubung. Mencoba mengambil satu titik data...")

    # Memilih satu titik waktu dan satu koordinat (misal: dekat Surabaya)
    single_point_data = ds['CRW_SST'].sel(
        time='2023-06-01T12:00:00', 
        latitude=-7.25, 
        longitude=112.75,
        method='nearest' # Ambil data dari grid terdekat
    )

    # .load() atau .values akan memicu unduhan data kecil ini
    sst_value = single_point_data.values
    
    print("\n" + "="*40)
    print("✅  KONEKSI BERHASIL!")
    print(f"Suhu Permukaan Laut (SST) di dekat Surabaya pada 1 Juni 2023 adalah: {sst_value}°C")
    print("="*40)

except Exception as e:
    print("\n" + "="*40)
    print("❌  TES KONEKSI GAGAL")
    print(f"Terjadi error: {e}")
    print("="*40)

Mencoba menghubungkan ke server...
Berhasil terhubung. Mencoba mengambil satu titik data...

✅  KONEKSI BERHASIL!
Suhu Permukaan Laut (SST) di dekat Surabaya pada 1 Juni 2023 adalah: 28.76°C


Data Acquisition and Preprocessing

In [5]:
import xarray as xr
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import sys
from tqdm import tqdm
import pandas as pd
import os
import glob

# --- Parameter Konfigurasi ---
BBOX = {
    "min_lon": 95.0, "max_lon": 141.0,
    "min_lat": -11.0, "max_lat": 6.0,
}
VARIABLES = ['CRW_SST', 'CRW_SSTANOMALY', 'CRW_HOTSPOT', 'CRW_DHW']
TIME_STEPS = 12
YEARS_TO_DOWNLOAD = [2024]
DATA_URL = "https://pae-paha.pacioos.hawaii.edu/thredds/dodsC/dhw_5km"
# Folder untuk menyimpan file bulanan
MONTHLY_DATA_DIR = "monthly_data"

# --- Fungsi ---
def create_sequences(data, n_steps):
    X, y = [], []
    iterator = tqdm(range(len(data)), desc="Membuat Sekuens Data")
    for i in iterator:
        end_ix = i + n_steps
        if end_ix >= len(data): break
        seq_x, seq_y = data[i:end_ix], data[end_ix]
        X.append(seq_x)
        y.append(seq_y)
    return np.array(X), np.array(y)

# --- Alur Utama ---
# ==============================================================================
# BAGIAN 1: PENGUNDUH BULANAN DENGAN CHECKPOINT
# ==============================================================================

# Buat folder jika belum ada
os.makedirs(MONTHLY_DATA_DIR, exist_ok=True)

for year in YEARS_TO_DOWNLOAD:
    for month in range(1, 13):
        # Tentukan nama file lokal untuk bulan ini
        file_path = os.path.join(MONTHLY_DATA_DIR, f"data_{year}_{month:02d}.npy")
            
        # Periksa apakah file sudah ada
        if os.path.exists(file_path):
            print(f"✅ File {file_path} sudah ada, melompati unduhan.")
            continue

        # Jika belum ada, lakukan proses unduh
        start_date = f"{year}-{month:02d}-01"
        end_of_month = pd.to_datetime(start_date) + pd.offsets.MonthEnd(0)
        end_date = end_of_month.strftime('%Y-%m-%d')
            
        print(f"\n--- Mengunduh data untuk: {start_date} hingga {end_date} ---")
            
        try:
            dataset = xr.open_dataset(DATA_URL, engine='netcdf4').sel(
                time=slice(start_date, end_date),
                longitude=slice(BBOX['min_lon'], BBOX['max_lon']),
                latitude=slice(BBOX['max_lat'], BBOX['min_lat'])
            )
                
            data_array = dataset[VARIABLES].to_array().values
            data_array = np.moveaxis(data_array, 0, -1)
            data_array = np.nan_to_num(data_array)
                
            # Simpan data bulan ini secara lokal
            np.save(file_path, data_array)
            print(f"💾 Data berhasil diunduh dan disimpan di {file_path}")

        except Exception as e:
            print(f"❌ Gagal mengunduh data untuk bulan {year}-{month:02d}. Error: {e}")
            print("Silakan jalankan ulang skrip untuk mencoba lagi.")
            sys.exit() # Hentikan skrip jika ada error unduh

print("\n🎉 Semua data bulanan telah berhasil diunduh.")

✅ File monthly_data\data_2024_01.npy sudah ada, melompati unduhan.
✅ File monthly_data\data_2024_02.npy sudah ada, melompati unduhan.
✅ File monthly_data\data_2024_03.npy sudah ada, melompati unduhan.
✅ File monthly_data\data_2024_04.npy sudah ada, melompati unduhan.

--- Mengunduh data untuk: 2024-05-01 hingga 2024-05-31 ---
💾 Data berhasil diunduh dan disimpan di monthly_data\data_2024_05.npy

--- Mengunduh data untuk: 2024-06-01 hingga 2024-06-30 ---
💾 Data berhasil diunduh dan disimpan di monthly_data\data_2024_06.npy

--- Mengunduh data untuk: 2024-07-01 hingga 2024-07-31 ---
💾 Data berhasil diunduh dan disimpan di monthly_data\data_2024_07.npy

--- Mengunduh data untuk: 2024-08-01 hingga 2024-08-31 ---
💾 Data berhasil diunduh dan disimpan di monthly_data\data_2024_08.npy

--- Mengunduh data untuk: 2024-09-01 hingga 2024-09-30 ---
💾 Data berhasil diunduh dan disimpan di monthly_data\data_2024_09.npy

--- Mengunduh data untuk: 2024-10-01 hingga 2024-10-31 ---
💾 Data berhasil diundu

In [5]:
import xarray as xr
import numpy as np
import geopandas as gpd
from sklearn.preprocessing import MinMaxScaler
import sys
from tqdm import tqdm
import os
import glob
import pandas as pd
import rioxarray

# --- Parameter Konfigurasi ---
BBOX = {
    "min_lon": 95.0, "max_lon": 141.0,
    "min_lat": -11.0, "max_lat": 6.0,
}
VARIABLES = ['CRW_SST', 'CRW_SSTANOMALY', 'CRW_HOTSPOT', 'CRW_DHW']
TIME_STEPS = 12
MONTHLY_DATA_DIR = "monthly_data"
SHAPEFILE_PATH = "Peta Terumbu Karang Indonesia/Coral_Ind_250K.shp"
FINAL_DATA_FILE = "processed_masked_data.npz"

# --- Fungsi ---
def create_sequences(data, n_steps):
    X, y = [], []
    iterator = tqdm(range(len(data)), desc="Membuat Sekuens Data")
    for i in iterator:
        end_ix = i + n_steps
        if end_ix >= len(data): break
        seq_x, seq_y = data[i:end_ix], data[end_ix]
        X.append(seq_x)
        y.append(seq_y)
    print("\nMengonversi list ke NumPy array dengan tipe float32...")
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

# --- Alur Utama ---
try:
    print(f"Memuat peta lokasi terumbu karang dari: {SHAPEFILE_PATH}")
    coral_map = gpd.read_file(SHAPEFILE_PATH)

    print("\nMemuat file bulanan dan melakukan 'masking'...")
    all_masked_data = []
    file_list = sorted(glob.glob(os.path.join(MONTHLY_DATA_DIR, "*.npy")))

    if not file_list:
        print(f"Error: Tidak ada file .npy yang ditemukan di folder '{MONTHLY_DATA_DIR}'.")
        sys.exit()

    for file_path in tqdm(file_list, desc="Memproses & Masking File Bulanan"):
        monthly_array = np.load(file_path)
        
        num_days, num_lats, num_lons, num_vars = monthly_array.shape
        lats = np.linspace(BBOX['max_lat'], BBOX['min_lat'], num_lats)
        lons = np.linspace(BBOX['min_lon'], BBOX['max_lon'], num_lons)
        
        ds_monthly = xr.Dataset(
            {var: (("time", "latitude", "longitude"), monthly_array[:, :, :, i]) 
             for i, var in enumerate(VARIABLES)},
            coords={
                "time": pd.to_datetime(range(num_days), unit='D'),
                "latitude": lats,
                "longitude": lons,
            }
        )
        
        ds_monthly = ds_monthly.rio.set_spatial_dims(x_dim="longitude", y_dim="latitude")
        ds_monthly = ds_monthly.rio.write_crs(coral_map.crs)
        clipped_data = ds_monthly.rio.clip(coral_map.geometry, all_touched=True)
        
        clipped_array = clipped_data[VARIABLES].to_array().values
        clipped_array = np.moveaxis(clipped_array, 0, -1)
        all_masked_data.append(clipped_array)

    print("\nMenggabungkan semua data bulanan yang sudah di-mask...")
    full_data_array = np.concatenate(all_masked_data, axis=0)
    full_data_array = np.nan_to_num(full_data_array)

    print("\nMelakukan normalisasi data...")
    scaler = MinMaxScaler()
    original_shape = full_data_array.shape
    # PERBAIKAN: Menggunakan -1 (satu minus) bukan --1
    data_flat = full_data_array.astype(np.float32).reshape(-1, full_data_array.shape[-1])
    data_scaled = scaler.fit_transform(data_flat)
    data_scaled = data_scaled.reshape(original_shape)
    print(f"Bentuk data gabungan setelah diproses: {data_scaled.shape}")

    X, y = create_sequences(data_scaled, TIME_STEPS)
    y = y[..., 3:4]

    print(f"Bentuk input (X) final: {X.shape}")
    print(f"Bentuk output (y) final: {y.shape}")

    print(f"\nMenyimpan data final yang telah diproses ke '{FINAL_DATA_FILE}'...")
    np.savez_compressed(FINAL_DATA_FILE, X=X, y=y)
    
    print("\n🚀 Pra-pemrosesan data dengan filter peta berhasil diselesaikan!")

except Exception as e:
    print(f"\nTerjadi Error: {e}")
    sys.exit()

Memuat peta lokasi terumbu karang dari: Peta Terumbu Karang Indonesia/Coral_Ind_250K.shp

Memuat file bulanan dan melakukan 'masking'...


Memproses & Masking File Bulanan: 100%|██████████| 12/12 [01:34<00:00,  7.83s/it]



Menggabungkan semua data bulanan yang sudah di-mask...

Melakukan normalisasi data...
Bentuk data gabungan setelah diproses: (361, 338, 914, 4)


Membuat Sekuens Data:  97%|█████████▋| 349/361 [00:00<?, ?it/s]


Mengonversi list ke NumPy array dengan tipe float32...


Bentuk input (X) final: (349, 12, 338, 914, 4)
Bentuk output (y) final: (349, 338, 914, 1)

Menyimpan data final yang telah diproses ke 'processed_masked_data.npz'...

🚀 Pra-pemrosesan data dengan filter peta berhasil diselesaikan!
